# FAERS cross-quarter profiling — guided notebook

This notebook walks through the same cross-quarter questions as `profile_cross_quarter.py`, one stage at a time. Run cells from top to bottom with **Shift+Enter**; each section explains what it is checking before it runs.

**Important:** this is a profiling exercise, not a cleaning pipeline. It reads the existing per-quarter DuckDB databases in read-only mode. It does not read or change the ZIP/raw text files, alter the quarter databases, or create an analysis-ready dataset. The “latest version, remove deleted cases” table is only a candidate rule to examine.

If a cell takes a while, let it finish before moving on. Combining all quarters and the drug-table comparisons can take a few minutes.

## 0. What you need and how to open this notebook

Open this file in VS Code or Jupyter from the project folder. Select the project's `.venv` Python interpreter as the notebook kernel (in VS Code, use **Select Kernel** at the top right). The project already needs `duckdb` installed; `pandas` is used to display query results. If imports fail, open a terminal in the project folder and run `python -m pip install -r requirements.txt`, then select the `.venv` kernel and restart it.

The next cell checks the current working folder and finds the pre-built databases. Open Jupyter from this project folder so it can find `outputs`.

In [1]:
from pathlib import Path
import re
import duckdb
import pandas as pd
from IPython.display import display

# Open Jupyter from the project folder so this works on your computer and after cloning.
PROJECT_DIR = Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / "outputs"
assert OUTPUTS_DIR.is_dir(), f"Cannot find outputs folder: {OUTPUTS_DIR}"

# Each quarter database is inside its matching profile_XXQX folder.
db_files = sorted(
    folder / f"{folder.name}.duckdb"
    for folder in OUTPUTS_DIR.glob("profile_??Q?")
    if (folder / f"{folder.name}.duckdb").is_file()
)
[(p.parent.name, f"{p.stat().st_size / (1024**2):.1f} MB") for p in db_files]

[('profile_24Q3', '132.8 MB'),
 ('profile_24Q4', '138.5 MB'),
 ('profile_25Q1', '135.8 MB'),
 ('profile_25Q2', '132.3 MB'),
 ('profile_25Q3', '146.8 MB'),
 ('profile_25Q4', '131.0 MB'),
 ('profile_26Q1', '132.5 MB'),
 ('profile_26Q2', '122.0 MB')]

That last result should list the quarter databases (for example `profile_24Q3.duckdb`). These databases were produced in the earlier, single-quarter profiling step. This notebook will attach each one as read-only; it will not open the raw archives.

In [2]:
if not db_files:
    raise FileNotFoundError(f"No quarter profile databases found under {OUTPUTS_DIR}")

databases = {}
for path in db_files:
    match = re.fullmatch(r"profile_(\d{2}Q[1-4])", path.stem)
    if match:
        databases[match.group(1)] = path.resolve()

quarters = sorted(databases)
print(f"Quarters available: {', '.join(quarters)}")
print(f"Database count: {len(quarters)}")

Quarters available: 24Q3, 24Q4, 25Q1, 25Q2, 25Q3, 25Q4, 26Q1, 26Q2
Database count: 8


## 1. Connect and check what is inside

DuckDB is the database format used for the previous profiling results. We create a temporary, in-memory database for this notebook, then attach the existing quarter databases with `READ_ONLY`. The memory limit matches the original script. The small table listing below checks the tables available in each quarter.

In [3]:
con = duckdb.connect(":memory:")
con.execute("SET memory_limit='4GB'")
con.execute("SET threads TO 4")

for quarter, path in databases.items():
    alias = f"db_{quarter}"
    safe_path = path.as_posix().replace("'", "''")
    con.execute(f"ATTACH '{safe_path}' AS {alias} (READ_ONLY)")

table_listing = []
for quarter in quarters:
    table_listing.append(
        con.execute(
            f"SELECT '{quarter}' AS quarter, table_name, table_type "
            "FROM information_schema.tables "
            f"WHERE table_catalog = 'db_{quarter}' AND table_schema = 'main' "
            "ORDER BY table_name"
        ).df()
    )
display(pd.concat(table_listing, ignore_index=True))

,quarter,table_name,table_type
0,24Q3,del_ids,BASE TABLE
1,24Q3,demo,BASE TABLE
2,24Q3,demo_x,VIEW
3,24Q3,drug,BASE TABLE
4,24Q3,indi,BASE TABLE
...,...,...,...
67,26Q2,indi,BASE TABLE
68,26Q2,outc,BASE TABLE
69,26Q2,reac,BASE TABLE
70,26Q2,rpsr,BASE TABLE


## 2. Make a case-version table for comparison

The `DEMO` table is the case-level table. We take only the fields needed for this cross-quarter review and append each quarter to one temporary table called `case_q`. `q` records which quarter database the row came from; `q_idx` gives the quarters a chronological sort order. `ver` is `caseversion` converted to an integer where possible.

We also copy the `del_ids` table from the earliest quarter database. In the earlier profiling process, that table was built from the delete files found in the raw-data folder. Nothing is deleted here.

In [4]:
for index, quarter in enumerate(quarters, start=1):
    select = f"""
        SELECT {index} AS q_idx, '{quarter}' AS q,
               primaryid, caseid, try_cast(caseversion AS INTEGER) AS ver,
               init_fda_dt, fda_dt, event_dt, rept_cod, mfr_sndr, sex, age,
               occp_cod, reporter_country, occr_country
        FROM db_{quarter}.demo
    """
    if index == 1:
        con.execute(f"CREATE TABLE case_q AS {select}")
    else:
        con.execute(f"INSERT INTO case_q {select}")

first_quarter = quarters[0]
last_quarter = quarters[-1]
con.execute(f"CREATE TABLE del_ids AS SELECT * FROM db_{first_quarter}.del_ids")

print(f"Combined DEMO rows: {con.execute('SELECT count(*) FROM case_q').fetchone()[0]:,}")
print(f"Delete-list rows copied: {con.execute('SELECT count(*) FROM del_ids').fetchone()[0]:,}")

Combined DEMO rows: 3,253,489
Delete-list rows copied: 41,529


## 3. Stage A — inventory

A row in `case_q` is one case record as published in one quarter. The same `caseid` can occur in multiple quarters when a follow-up version is published. Therefore, total rows and distinct case IDs are different counts. `primaryid` identifies a specific report/version and is not the same thing as the underlying case ID.

In [5]:
display(con.execute("""
    SELECT q AS quarter, count(*) AS case_version_rows,
           count(DISTINCT caseid) AS distinct_cases
    FROM case_q GROUP BY q ORDER BY q
""").df())

display(con.execute("""
    SELECT count(*) AS rows_across_quarters,
           count(DISTINCT caseid) AS distinct_cases,
           count(*) - count(DISTINCT caseid) AS repeat_appearances,
           count(DISTINCT primaryid) AS distinct_report_ids
    FROM case_q
""").df())

,quarter,case_version_rows,distinct_cases
0,24Q3,405513,405513
1,24Q4,410849,410849
2,25Q1,400514,400514
3,25Q2,393130,393130
4,25Q3,438512,438512
5,25Q4,385288,385288
6,26Q1,397224,397224
7,26Q2,422459,422458


,rows_across_quarters,distinct_cases,repeat_appearances,distinct_report_ids
0,3253489,2870985,382504,3253358


## 4. Stage B — how cases and versions recur

First, count how many quarter files each case appears in. Then compare each case’s version number with its previous appearance. A repeated version is not automatically an error: a record can be present again without a higher version. The tie check finds the same `(caseid, version)` published in more than one quarter. These are investigation clues, not cleaning decisions.

In [6]:
display(con.execute("""
    WITH appearances AS (
        SELECT caseid, count(*) AS quarter_count FROM case_q GROUP BY caseid
    )
    SELECT quarter_count, count(*) AS cases
    FROM appearances GROUP BY quarter_count ORDER BY quarter_count
""").df())

display(con.execute("""
    WITH ordered AS (
        SELECT caseid, ver,
               lag(ver) OVER (PARTITION BY caseid ORDER BY q_idx) AS previous_version
        FROM case_q
    )
    SELECT count(*) FILTER (WHERE previous_version IS NOT NULL) AS later_appearances,
           count(*) FILTER (WHERE ver > previous_version) AS version_increased,
           count(*) FILTER (WHERE ver = previous_version) AS same_version,
           count(*) FILTER (WHERE ver < previous_version) AS version_decreased
    FROM ordered
""").df())

display(con.execute("""
    SELECT count(*) AS case_version_pairs_repeated,
           coalesce(sum(n), 0) AS rows_in_those_pairs,
           count(*) FILTER (WHERE different_fda_dates > 1) AS pairs_with_different_fda_dates
    FROM (
        SELECT caseid, ver, count(*) AS n,
               count(DISTINCT fda_dt) AS different_fda_dates
        FROM case_q GROUP BY caseid, ver HAVING count(*) > 1
    )
""").df())

,quarter_count,cases
0,1,2556369
1,2,263005
2,3,39191
3,4,9318
4,5,2454
5,6,553
6,7,83
7,8,12


,later_appearances,version_increased,same_version,version_decreased
0,382504,382372,131,1


,case_version_pairs_repeated,rows_in_those_pairs,pairs_with_different_fda_dates
0,131,262.0,0


You can look at individual cases where the version did not increase, instead of relying only on a summary count. This is a sample of at most ten examples.

In [7]:
display(con.execute("""
    WITH ordered AS (
        SELECT caseid, q, ver, fda_dt,
               lag(q) OVER (PARTITION BY caseid ORDER BY q_idx) AS previous_quarter,
               lag(ver) OVER (PARTITION BY caseid ORDER BY q_idx) AS previous_version
        FROM case_q
    )
    SELECT caseid, previous_quarter, previous_version,
           q AS later_quarter, ver AS later_version, fda_dt
    FROM ordered
    WHERE previous_version IS NOT NULL AND ver <= previous_version
    ORDER BY caseid LIMIT 10
""").df())

,caseid,previous_quarter,previous_version,later_quarter,later_version,fda_dt
0,14704822,25Q1,20,25Q2,20,20250331
1,18302191,25Q1,4,25Q2,4,20250331
2,18724266,25Q1,47,25Q2,47,20250331
3,18997001,25Q1,19,25Q2,19,20250331
4,20629819,25Q1,4,25Q2,4,20250331
5,23244838,25Q1,11,25Q2,11,20250331
6,24773801,25Q1,3,25Q2,3,20250331
7,24776847,25Q1,3,25Q2,3,20250331
8,24790685,25Q1,3,25Q2,3,20250331
9,24790692,25Q1,2,25Q2,2,20250331


## 5. Stage C — simulate a possible latest-version rule

For this demonstration, choose one row per `caseid`: highest version first, then latest `fda_dt`, then latest quarter as a tie-breaker. Next, exclude IDs present in the delete list. The result is a **candidate case-level set**, not an official or validated final dataset. It only chooses rows from `DEMO`; it has not yet selected or joined the matching `DRUG`, `REAC`, `INDI`, `OUTC`, `THER`, or `RPSR` records.

In [8]:
con.execute("""
    CREATE TABLE winners AS
    SELECT * EXCLUDE (row_number_in_case)
    FROM (
        SELECT *, row_number() OVER (
            PARTITION BY caseid
            ORDER BY ver DESC NULLS LAST, fda_dt DESC, q_idx DESC
        ) AS row_number_in_case
        FROM case_q
    )
    WHERE row_number_in_case = 1
""")
con.execute("""
    CREATE TABLE candidate_cases AS
    SELECT * FROM winners
    WHERE caseid NOT IN (SELECT caseid FROM del_ids)
""")

display(con.execute("""
    SELECT (SELECT count(*) FROM case_q) AS all_quarter_rows,
           (SELECT count(*) FROM winners) AS one_row_per_case,
           (SELECT count(*) FROM winners WHERE caseid IN (SELECT caseid FROM del_ids)) AS matched_delete_ids,
           (SELECT count(*) FROM candidate_cases) AS remaining_candidate_cases
""").df())

display(con.execute("""
    SELECT q AS quarter_of_selected_version, count(*) AS cases
    FROM candidate_cases GROUP BY q ORDER BY q
""").df())

,all_quarter_rows,one_row_per_case,matched_delete_ids,remaining_candidate_cases
0,3253489,2870985,12912,2858073


,quarter_of_selected_version,cases
0,24Q3,342975
1,24Q4,333636
2,25Q1,325021
3,25Q2,326905
4,25Q3,391622
5,25Q4,345568
6,26Q1,369888
7,26Q2,422458


## 6. Stage D — compare delete-list timing with publication

This asks whether a case found in a delete list also appears in these quarter databases before or after the quarter of its first delete listing. The list is the delete data already stored in the earliest quarter database. This comparison helps check assumptions about when a deleted case should be excluded.

In [9]:
display(con.execute("""
    WITH delete_quarter AS (
        SELECT caseid, min(src) AS first_delete_quarter
        FROM del_ids GROUP BY caseid
    ), appearances AS (
        SELECT caseid, max(q) AS last_seen_quarter, min(q) AS first_seen_quarter
        FROM case_q GROUP BY caseid
    )
    SELECT count(*) AS deleted_cases_seen_in_window,
           count(*) FILTER (WHERE first_seen_quarter < first_delete_quarter) AS seen_before_delete_listing,
           count(*) FILTER (WHERE last_seen_quarter = first_delete_quarter) AS last_seen_same_quarter,
           count(*) FILTER (WHERE last_seen_quarter > first_delete_quarter) AS seen_in_later_quarter
    FROM appearances JOIN delete_quarter USING (caseid)
""").df())

,deleted_cases_seen_in_window,seen_before_delete_listing,last_seen_same_quarter,seen_in_later_quarter
0,12912,12911,1,0


## 7. Stage E — check data completeness and how it shifts

These percentages measure fields that are blank/NULL in `DEMO`. Missing does not necessarily mean invalid; it means an analysis must account for unavailable information. The second table breaks missing occupation down by report type so you can see which report groups contribute to the change.

In [10]:
display(con.execute("""
    SELECT q AS quarter, count(*) AS rows,
           round(100.0 * count(*) FILTER (WHERE occp_cod IS NULL) / count(*), 1) AS pct_occupation_null,
           round(100.0 * count(*) FILTER (WHERE reporter_country = 'COUNTRY NOT SPECIFIED') / count(*), 1) AS pct_reporter_country_unspecified,
           round(100.0 * count(*) FILTER (WHERE occr_country IS NULL) / count(*), 1) AS pct_event_country_null,
           round(100.0 * count(*) FILTER (WHERE sex IS NULL) / count(*), 1) AS pct_sex_null,
           round(100.0 * count(*) FILTER (WHERE age IS NULL) / count(*), 1) AS pct_age_null,
           round(100.0 * count(*) FILTER (WHERE event_dt IS NULL) / count(*), 1) AS pct_event_date_null
    FROM case_q GROUP BY q ORDER BY q
""").df())

display(con.execute("""
    SELECT q AS quarter, rept_cod AS report_type, count(*) AS rows,
           round(100.0 * count(*) FILTER (WHERE occp_cod IS NULL) / count(*), 1) AS pct_occupation_null
    FROM case_q GROUP BY q, rept_cod ORDER BY q, rept_cod
""").df())

display(con.execute(f"""
    SELECT mfr_sndr AS sender, rept_cod AS report_type, count(*) AS rows_with_missing_occupation
    FROM case_q WHERE q = '{last_quarter}' AND occp_cod IS NULL
    GROUP BY mfr_sndr, rept_cod ORDER BY rows_with_missing_occupation DESC LIMIT 10
""").df())

,quarter,rows,pct_occupation_null,pct_reporter_country_unspecified,pct_event_country_null,pct_sex_null,pct_age_null,pct_event_date_null
0,24Q3,405513,2.1,0.2,7.4,17.1,40.9,55.6
1,24Q4,410849,5.8,0.1,3.8,20.0,39.2,55.6
2,25Q1,400514,8.2,0.1,4.0,20.4,39.2,56.9
3,25Q2,393130,10.1,0.2,4.4,21.5,39.7,55.7
4,25Q3,438512,17.1,0.1,3.8,17.0,41.3,57.1
5,25Q4,385288,20.0,0.2,7.6,20.5,38.4,55.3
6,26Q1,397224,35.1,1.0,8.1,19.0,38.2,57.7
7,26Q2,422459,72.1,2.4,11.6,18.8,37.0,58.2


,quarter,report_type,rows,pct_occupation_null
0,24Q3,30DAY,10133,0.1
1,24Q3,5DAY,634,0.0
2,24Q3,DIR,13485,36.2
3,24Q3,EXP,202860,0.9
4,24Q3,PER,178401,1.0
5,24Q4,30DAY,8295,0.1
6,24Q4,5DAY,672,1.8
7,24Q4,DIR,14470,31.8
8,24Q4,EXP,211577,6.7
9,24Q4,PER,175835,2.9


,sender,report_type,rows_with_missing_occupation
0,SANOFI AVENTIS,PER,47213
1,ELI LILLY AND CO,PER,19030
2,ABBVIE,EXP,14354
3,NOVO NORDISK,PER,9717
4,AMGEN,EXP,8757
5,ALEXION PHARMACEUTICALS,EXP,7554
6,BRISTOL-MYERS SQUIBB COMPANY,PER,6720
7,UCB,PER,6174
8,SANOFI AVENTIS,EXP,5140
9,ROCHE,EXP,4531


## 8. Stage F — inspect drug-table row patterns

`DRUG` is a detail table: one report can have several drug rows. The first comparison shows raw drug rows, distinct `(primaryid, drug_seq)` combinations, and counts for primary-suspect (`PS`) rows. The ratios help describe how the table changes; they do not prove data corruption or tell us to drop rows.

In [11]:
drug_summary_parts = []
for quarter in quarters:
    drug_summary_parts.append(f"""
        SELECT '{quarter}' AS quarter,
               count(*) AS drug_rows,
               count(DISTINCT primaryid || '|' || drug_seq) AS distinct_case_sequences,
               round(1.0 * count(*) / count(DISTINCT primaryid), 2) AS rows_per_report,
               count(*) FILTER (WHERE role_cod = 'PS') AS primary_suspect_rows,
               count(DISTINCT primaryid || '|' || drug_seq) FILTER (WHERE role_cod = 'PS') AS primary_suspect_sequences,
               count(DISTINCT primaryid || '|' || upper(trim(prod_ai))) FILTER (WHERE role_cod = 'PS') AS primary_suspect_report_ingredients
        FROM db_{quarter}.drug
    """)

drug_summary_sql = " UNION ALL ".join(drug_summary_parts) + " ORDER BY quarter"
display(con.execute(drug_summary_sql).df())

,quarter,drug_rows,distinct_case_sequences,rows_per_report,primary_suspect_rows,primary_suspect_sequences,primary_suspect_report_ingredients
0,24Q3,1907293,1903347,4.70,408860,405512,405509
1,24Q4,2030938,1994783,4.94,434165,410849,410848
2,25Q1,2008162,1961409,5.01,431702,400514,400514
3,25Q2,1829056,1766389,4.65,428772,393130,393129
4,25Q3,2148451,2067596,4.90,480527,438512,438512
5,25Q4,1815349,1675771,4.71,437206,385278,385278
6,26Q1,1703210,1501182,4.29,463454,397209,397209
7,26Q2,1627225,1324647,3.85,530352,422396,422396


The next check looks at repeated `(primaryid, drug_seq)` groups and counts groups where values disagree for selected fields. Repeated sequence groups can carry different dose/route/lot details. Review them before choosing any deduplication rule.

In [12]:
repeat_parts = []
for quarter in quarters:
    repeat_parts.append(f"""
        SELECT '{quarter}' AS quarter,
               count(*) AS repeated_sequence_groups,
               count(*) FILTER (WHERE drug_names > 1) AS groups_with_different_drug_names,
               count(*) FILTER (WHERE ingredients > 1) AS groups_with_different_ingredients,
               count(*) FILTER (WHERE roles > 1) AS groups_with_different_roles,
               count(*) FILTER (WHERE routes > 1) AS groups_with_different_routes,
               count(*) FILTER (WHERE doses > 1) AS groups_with_different_doses,
               count(*) FILTER (WHERE frequencies > 1) AS groups_with_different_frequencies,
               count(*) FILTER (WHERE lots > 1) AS groups_with_different_lots
        FROM (
            SELECT primaryid, drug_seq,
                   count(DISTINCT drugname) AS drug_names,
                   count(DISTINCT prod_ai) AS ingredients,
                   count(DISTINCT role_cod) AS roles,
                   count(DISTINCT route) AS routes,
                   count(DISTINCT dose_amt) AS doses,
                   count(DISTINCT dose_freq) AS frequencies,
                   count(DISTINCT lot_num) AS lots
            FROM db_{quarter}.drug
            GROUP BY primaryid, drug_seq HAVING count(*) > 1
        )
    """)

repeat_sql = " UNION ALL ".join(repeat_parts) + " ORDER BY quarter"
display(con.execute(repeat_sql).df())

,quarter,repeated_sequence_groups,groups_with_different_drug_names,groups_with_different_ingredients,groups_with_different_roles,groups_with_different_routes,groups_with_different_doses,groups_with_different_frequencies,groups_with_different_lots
0,24Q3,2267,0,0,0,0,738,6,62
1,24Q4,18698,0,0,0,305,3603,636,2634
2,25Q1,21797,0,0,0,437,5876,924,3975
3,25Q2,31391,0,0,0,590,7540,1221,4380
4,25Q3,43481,0,0,0,831,11783,1527,5401
5,25Q4,62384,0,0,0,4375,20370,2491,6700
6,26Q1,87928,0,0,0,8111,32261,3492,9335
7,26Q2,147113,0,0,0,10571,56582,5545,16466


Finally, this checks how many distinct primary-suspect ingredients are recorded per report. The output is a descriptive check only; it does not determine how ingredients should be normalized or interpreted.

In [13]:
suspect_parts = []
for quarter in quarters:
    suspect_parts.append(f"""
        SELECT '{quarter}' AS quarter,
               count(*) AS reports_with_primary_suspect,
               round(100.0 * count(*) FILTER (WHERE ingredient_count = 1) / count(*), 1) AS pct_one_ingredient,
               round(100.0 * count(*) FILTER (WHERE ingredient_count = 2) / count(*), 1) AS pct_two_ingredients,
               round(100.0 * count(*) FILTER (WHERE ingredient_count >= 3) / count(*), 1) AS pct_three_or_more
        FROM (
            SELECT primaryid, count(DISTINCT upper(trim(prod_ai))) AS ingredient_count
            FROM db_{quarter}.drug WHERE role_cod = 'PS' GROUP BY primaryid
        )
    """)

suspect_sql = " UNION ALL ".join(suspect_parts) + " ORDER BY quarter"
display(con.execute(suspect_sql).df())

,quarter,reports_with_primary_suspect,pct_one_ingredient,pct_two_ingredients,pct_three_or_more
0,24Q3,405512,100.0,0.0,0.0
1,24Q4,410849,100.0,0.0,0.0
2,25Q1,400514,100.0,0.0,0.0
3,25Q2,393130,100.0,0.0,0.0
4,25Q3,438512,100.0,0.0,0.0
5,25Q4,385278,100.0,0.0,0.0
6,26Q1,397209,100.0,0.0,0.0
7,26Q2,422396,100.0,0.0,0.0


## 9. Where we are and what should follow

At this point you have inspected case counts and version behavior, simulated a possible case-selection rule, checked deletion timing and missingness, and examined drug-row patterns. You have **not** cleaned, normalized, or joined the seven FAERS tables.

A sensible next project stage is to agree on and document the analysis rules, then build a separate transformation that applies the chosen case/version/deletion logic consistently to `DEMO` and its related detail tables. Validate row counts and join keys before using that result in the dashboard. FAERS reports describe suspected adverse events; they do not by themselves establish that a drug caused an event.

You can keep exploring by writing more `SELECT` queries in new cells. When finished, close this temporary connection (the source databases remain read-only).

In [14]:
con.close()
print("Notebook's in-memory DuckDB connection closed. Source databases were read-only.")

Notebook's in-memory DuckDB connection closed. Source databases were read-only.
